# Working with Django Models — Complete Guide

This notebook covers the full Django ORM and model layer from first principles to advanced customization. Uses Django 5 and Python 3.12 (applicable to Django 4.x / 5.x, Python 3.10+).

**Modules covered:**
1. Introduction: ORM and Django Models
2. Django Model Classes and Instances
3. Django Model Fields
4. Managers and QuerySets
5. Customizing Model Behaviour
6. Django Migrations
7. Optimizing the ORM

---

## 1. Introduction: ORM and Django Models

### 1.1 Django's Architecture: Model-Template-View

Django follows the **MTV** (Model-Template-View) pattern, which maps to MVC:

| Layer | Responsibility | Django component |
|---|---|---|
| **Model** | Data | Python class mapped to a DB table |
| **Template** | Presentation | HTML generator |
| **View** | Behaviour | Python function mapped to a URL |

### 1.2 What is the ORM?

**ORM = Object-Relational Mapper** — another name for Django's model layer.

| Relational world | Python world |
|---|---|
| Rows in tables | Instances of classes |
| SQL | Python code |

The ORM bridges the gap:
- Model classes are mapped to tables
- SQL is generated automatically
- You only write Python

**Disadvantages of the ORM:**
- Less control over exact SQL generated
- Slight performance overhead

**But:** you can optimize queries with Django's built-in tools, and you can always drop down to raw SQL when needed.

### 1.3 What Django Models Do

A Django model is more than just a database table. It:
- Is **mapped to a DB table** (primary use)
- Can **generate UI** via `ModelForm`
- **Validates user input** via forms
- **Generates the admin interface** automatically
- Can carry **custom methods** (business logic)

### 1.4 Supported Databases

Built-in: PostgreSQL, MariaDB, MySQL, Oracle, SQLite  
Via packages: DB2, MS SQL, and more

---
## 2. Django Model Classes and Instances

### 2.1 Models and Migrations

| Concept | What it is |
|---|---|
| **Model** | Python class mapped to a DB table; each object is a row |
| **Migration** | Python script that keeps DB structure in sync with your model code |

Migrations are auto-generated (but not always — sometimes you need to write them manually).

### 2.2 Defining a Model Class

In [ ]:
# models.py
from django.db import models

class Person(models.Model):
    name = models.CharField(max_length=100)
    age = models.IntegerField()

- Every model class inherits from `models.Model`
- Each field is a class attribute and maps to a database column
- Django auto-adds a primary key field `id` (auto-increment integer) unless you define one

### 2.3 The Migration Workflow

Always follow this 3-step process when changing model code:

```
# Important: Make sure your app is in INSTALLED_APPS first!

# Step 1: Change your model code

# Step 2: Generate migration script (always check the generated file!)
python manage.py makemigrations

# Optional: Preview pending migrations
python manage.py showmigrations

# Step 3: Apply migrations to the database
python manage.py migrate
```

### 2.4 Creating, Saving, and Deleting Instances

In [ ]:
# Create a new instance in memory (not yet in DB)
p = Product(name="Blue Socks", price=4.99, stock_count=100)

# save() does SQL INSERT for a new object and sets the id field
# or SQL UPDATE for an existing object with changes
p.save()

print(p.id)   # now has a PK assigned

# Update: change an attribute and save again
p.price = 3.99
p.save()       # SQL UPDATE

# Delete the object from the database
p.delete()     # SQL DELETE

**Key point:** `save()` is smart — if `id` is `None` (new object) it does an INSERT; if `id` already exists it does an UPDATE.

### 2.5 Different Backends

The same model code works with any supported DB backend. Django generates the correct SQL dialect for each. You can also generate Django model classes from an existing database using `python manage.py inspectdb`.

---
## 3. Django Model Fields

### 3.1 What Are Model Fields?

Model fields are **class attributes** that:
- Map to **database columns**
- Must be instances of a **Field class** (e.g., `IntegerField`, `CharField`)
- Determine the **DB column type** (INTEGER, VARCHAR, ...)
- Control **how the field is rendered** in a form
- Carry **validation rules**

### 3.2 Numeric Fields

In [ ]:
from django.db import models

class Product(models.Model):
    # Integer types
    stock_count = models.IntegerField()          # standard integer
    # Also: SmallIntegerField, BigIntegerField, PositiveIntegerField, etc.

    # Float — IEEE 754, has precision issues
    weight = models.FloatField()

    # Decimal — exact precision, use for money!
    price = models.DecimalField(max_digits=10, decimal_places=2)

    # Boolean
    is_active = models.BooleanField(default=True)

> **Critical Rule:** Never store currency / money as a `FloatField`. Floating-point numbers have binary representation errors:
>
> ```python
> price = 0.1
> price *= 3
> print(price)  # 0.30000000000000004  ← wrong!
> ```
>
> Always use `DecimalField` for prices.

### 3.3 Text Fields

In [ ]:
class Article(models.Model):
    # CharField: fixed-length text, renders as <input type="text">
    # max_length is REQUIRED
    title = models.CharField(max_length=200)
    slug = models.SlugField(max_length=200)         # URL-friendly text

    # TextField: long-form text, renders as <textarea>
    body = models.TextField()

    # Specialised text variants (still stored as VARCHAR/TEXT)
    email = models.EmailField()                     # validated as email
    website = models.URLField()                     # validated as URL
    config_path = models.FilePathField(path='/etc') # file system path
    ip_address = models.GenericIPAddressField()     # IPv4 or IPv6

### 3.4 Date and Time Fields

In [ ]:
class Event(models.Model):
    date_only = models.DateField()
    time_only = models.TimeField()
    timestamp = models.DateTimeField()
    duration  = models.DurationField()             # Python timedelta

    # auto_now_add: set once on creation; auto_now: update on every save
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

### 3.5 Other Common Field Types

In [ ]:
class Document(models.Model):
    # File / image uploads
    attachment = models.FileField(upload_to='documents/')
    thumbnail  = models.ImageField(upload_to='images/')  # requires Pillow

    # Structured / binary data
    metadata   = models.JSONField(default=dict)
    raw_data   = models.BinaryField()

### 3.6 Field Options (Common to All Fields)

**Group 1: Null / blank / default**

In [ ]:
# null=True  → column allows NULL in the database (default: NOT NULL)
discount = models.IntegerField(null=True)

# blank=True → form validation allows empty value (NOT a DB constraint!)
bio = models.CharField(max_length=500, blank=True)

# default → value used when field is not provided
status = models.CharField(max_length=20, default='active')

**Group 2: Constraints and indexing**

In [ ]:
# unique=True → database UNIQUE constraint
sku = models.CharField(max_length=20, unique=True)

# db_index=True → creates a database index (speeds up lookups)
category_id = models.IntegerField(db_index=True)

# db_column → override the column name in the DB
is_published = models.BooleanField(db_column='published_flag')

# Type-specific: DateTimeField.auto_now
updated = models.DateTimeField(auto_now=True)

**Group 3: Display / form options**

In [ ]:
# verbose_name → human-readable label shown in forms and admin
iban = models.CharField(max_length=34, verbose_name='Bank Account Number')

# help_text → hint shown below the form field
name = models.CharField(max_length=100, help_text='Enter your full name')

### 3.7 Relationships Between Models

#### ForeignKey (one-to-many)

In [ ]:
class ProductImage(models.Model):
    # ForeignKey creates a one-to-many relation.
    # Place it on the "many" side (many images belong to one product).
    product = models.ForeignKey(
        'Product',              # Use string name to avoid circular imports
        on_delete=models.CASCADE  # Required: what to do when Product is deleted
    )
    image_url = models.URLField()

# on_delete options:
# CASCADE    → delete child when parent is deleted
# PROTECT    → raise error if parent has children
# SET_NULL   → set FK to NULL (requires null=True)
# SET_DEFAULT → set FK to default value
# DO_NOTHING → do nothing (risky, may cause DB errors)

#### OneToOneField

In [ ]:
class UserProfile(models.Model):
    # Like ForeignKey with unique=True
    # Each user has exactly one profile
    user = models.OneToOneField(
        'auth.User',
        on_delete=models.CASCADE
    )
    bio = models.TextField(blank=True)

#### ManyToManyField

In [ ]:
class Category(models.Model):
    name = models.CharField(max_length=100)

    # ManyToManyField creates a join table in the DB automatically.
    # Can be placed on either side of the relation.
    products = models.ManyToManyField('Product')

# Usage:
cat = Category.objects.get(name='Socks')
cat.products.all()           # all products in this category
cat.products.add(product)    # add a product
cat.products.remove(product) # remove a product

---
## 4. Managers and QuerySets

### 4.1 The Manager

Every model has a default **Manager** accessible via `Model.objects`. It is the entry point for all database queries.

In [ ]:
# Count all products
Product.objects.count()

# Get a single product by primary key
Product.objects.get(pk=1)

# Get all products in the 'Swimwear' category
swimwear = Product.objects.filter(category__name='Swimwear')

### 4.2 QuerySets Are Lazy

A QuerySet does **not** execute SQL until it is evaluated. This lets you chain multiple filters efficiently — Django sends a single SQL query, not one per step.

In [ ]:
# No SQL yet — just building a query plan
in_stock = Product.objects.filter(stock_count__gt=0)

# Still no SQL — another filter appended
socks_in_stock = in_stock.filter(category__name='Socks')

# Still no SQL — slicing/indexing just adds LIMIT/OFFSET
cheapest_sock = socks_in_stock.order_by('price')[0]

# SQL is fired HERE, when the result is actually needed:
print(cheapest_sock.name)   # ← evaluation trigger

**QuerySet evaluation triggers** (when SQL is actually executed):
- Iterating over the queryset (`for p in qs:`)
- Calling `list(qs)` or `str(qs)`
- Slicing with a step (`qs[::2]`)
- Rendering in a Django template
- Calling `len()`, `bool()`, `repr()`

**QuerySet results are cached:** Once evaluated, the result set is stored — iterating the same queryset again does NOT hit the database.

### 4.3 Filtering: `filter()` and `exclude()`

In [ ]:
# filter() — returns objects MATCHING the conditions
Product.objects.filter(name__contains='sock')
Product.objects.filter(stock_count__gt=0, price__lt=10)

# exclude() — returns objects NOT matching the conditions
Product.objects.exclude(stock_count=0)           # all in-stock
Product.objects.exclude(stock_count=0, price=10) # not (out-of-stock AND price=10)

### 4.4 Field Lookups

Lookups use double-underscore syntax: `field__lookuptype=value`

| Lookup | Meaning | SQL |
|---|---|---|
| `exact` (default) | Exact match | `= value` |
| `iexact` | Case-insensitive exact | `ILIKE value` |
| `contains` | Case-sensitive substring | `LIKE '%value%'` |
| `icontains` | Case-insensitive substring | `ILIKE '%value%'` |
| `startswith` | Starts with | `LIKE 'value%'` |
| `endswith` | Ends with | `LIKE '%value'` |
| `gt` | Greater than | `> value` |
| `gte` | Greater than or equal | `>= value` |
| `lt` | Less than | `< value` |
| `lte` | Less than or equal | `<= value` |
| `in` | In a list | `IN (v1, v2, ...)` |
| `isnull` | Is NULL | `IS NULL` |
| `range` | In a range (inclusive) | `BETWEEN a AND b` |

In [ ]:
# Examples
Product.objects.filter(name__icontains='socks')
Product.objects.filter(price__gte=5, price__lte=20)
Product.objects.filter(category__name__in=['Socks', 'Shoes'])
Product.objects.filter(description__isnull=True)
Product.objects.filter(price__range=(5, 20))

### 4.5 Traversing Relations with `__`

The double-underscore `__` also traverses foreign key relationships across tables:

In [ ]:
# Get all ProductImages where:
#   - the related Product is in the 'Socks' category
#   - AND the related Product has stock_count > 0
ProductImage.objects.filter(
    product__category__name='Socks',
    product__stock_count__gt=0
)

# Django generates a SQL JOIN automatically

### 4.6 `get()` and `get_object_or_404()`

In [ ]:
from django.shortcuts import get_object_or_404

# get() — returns EXACTLY ONE object
# Raises DoesNotExist if none found
# Raises MultipleObjectsReturned if more than one matches
product = Product.objects.get(pk=5)
product = Product.objects.get(sku='abc005x')

# get_object_or_404() — like get(), but raises Http404 instead of DoesNotExist
# Use this in views so Django returns a proper 404 page
product = get_object_or_404(Product, pk=5)

---
## 5. Customizing Model Behaviour

### 5.1 The `Meta` Class

Define a nested `Meta` class inside your model to set model-level options:

In [ ]:
class Product(models.Model):
    name = models.CharField(max_length=200)
    price = models.DecimalField(max_digits=10, decimal_places=2)

    class Meta:
        # Human-readable singular name (shown in admin)
        verbose_name = 'Product'
        # Human-readable plural name
        verbose_name_plural = 'Products'
        # Default ordering when no .order_by() is specified
        ordering = ['name']          # ascending by name
        # ordering = ['-price']      # descending by price

        # Override the default DB table name
        db_table = 'store_product'

        # Multi-column unique constraints
        unique_together = [['name', 'sku']]

### 5.2 Custom Methods on Models: "Fat Models, Skinny Views"

Business logic belongs on the **model**, not scattered through views. This is the **fat models, skinny views** pattern:

In [ ]:
class Product(models.Model):
    name = models.CharField(max_length=200)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    stock_count = models.IntegerField(default=0)
    discount_percentage = models.IntegerField(default=0)

    # Custom property — no DB column, computed from other fields
    @property
    def discounted_price(self):
        return self.price * (1 - self.discount_percentage / 100)

    def is_in_stock(self):
        return self.stock_count > 0

    # __str__ controls how the object looks in admin and shell
    def __str__(self):
        return self.name

    # get_absolute_url — canonical URL for this object
    def get_absolute_url(self):
        from django.urls import reverse
        return reverse('product-detail', kwargs={'pk': self.pk})

### 5.3 Custom Managers

Custom managers let you add reusable query methods to your models. There are two approaches:

#### Approach 1: QuerySet subclass + `as_manager()` (preferred)

In [ ]:
class ProductQuerySet(models.QuerySet):
    def in_stock(self):
        return self.filter(stock_count__gt=0)

    def affordable(self, max_price):
        return self.filter(price__lte=max_price)


class Product(models.Model):
    name = models.CharField(max_length=200)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    stock_count = models.IntegerField(default=0)

    # Convert the QuerySet subclass into a Manager
    objects = ProductQuerySet.as_manager()


# Usage — methods are chainable!
cheap_socks_in_stock = (
    Product.objects
    .in_stock()
    .affordable(10)
    .filter(category__name='Socks')
)

#### Approach 2: Manager subclass + `get_queryset()` override

Use this when you want a manager that **always filters** the base queryset — e.g., soft-delete or tenant isolation:

In [ ]:
class ProductInStockManager(models.Manager):
    def get_queryset(self):
        # Every query through this manager pre-filters for stock > 0
        return super().get_queryset().filter(stock_count__gt=0)


class Product(models.Model):
    name = models.CharField(max_length=200)
    stock_count = models.IntegerField(default=0)

    # Keep the default manager for unrestricted access
    objects = models.Manager()
    # Add a second manager with automatic filtering
    in_stock = ProductInStockManager()


# Usage
Product.objects.all()     # ALL products
Product.in_stock.all()    # only products with stock_count > 0

### 5.4 Model Inheritance

Django offers three inheritance strategies:

#### Abstract Base Class

In [ ]:
class TimestampedModel(models.Model):
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        abstract = True   # No database table is created for this class


class Product(TimestampedModel):   # inherits fields, no extra table
    name = models.CharField(max_length=200)

class Order(TimestampedModel):     # also gets created_at / updated_at
    total = models.DecimalField(max_digits=10, decimal_places=2)

- **No separate table** for the base class
- Fields are copied into each child table
- Best for **shared fields** (timestamps, soft-delete flags, etc.)

#### Multi-Table Inheritance

In [ ]:
class Animal(models.Model):
    name = models.CharField(max_length=100)
    # Table: myapp_animal

class Dog(Animal):
    breed = models.CharField(max_length=100)
    # Table: myapp_dog
    # Has an implicit OneToOneField → Animal (implicit JOIN on access)

- **Each model gets its own table**
- Parent table stores common fields; child table stores extra fields
- Django does an automatic JOIN when you access the child
- Use sparingly — JOINs add overhead

#### Proxy Models

In [ ]:
class Product(models.Model):
    name = models.CharField(max_length=200)
    stock_count = models.IntegerField(default=0)

class InStockProduct(Product):
    class Meta:
        proxy = True    # Uses the SAME table as Product

    def save(self, *args, **kwargs):
        # Can override methods without changing the table
        if self.stock_count <= 0:
            raise ValueError('Cannot save an out-of-stock product here')
        super().save(*args, **kwargs)

- **Same table** as the parent
- Can add Python methods, custom managers, or change default ordering
- No new columns
- Useful for different admin views or different default behaviour on the same data

---
## 6. Django Migrations

### 6.1 What Are Migrations?

Migrations are **Python scripts** that keep the database structure in sync with your model code. They form a **dependency chain** — each migration knows which migrations it depends on.

Applied migrations are tracked in the database table `django_migrations`.

### 6.2 Inspecting Migrations

In [ ]:
# List all migrations and their applied status
python manage.py showmigrations

# List migrations for a specific app
python manage.py showmigrations my_app

# Show the SQL that a migration would run (without running it)
python manage.py sqlmigrate my_app 0005
# Note: migration names can be abbreviated to a unique prefix (e.g., 0005)

### 6.3 Creating Migrations

```bash
# Generate migrations for all apps
python manage.py makemigrations

# Generate migrations for a specific app
python manage.py makemigrations my_app

# Preview what would be generated (dry run)
python manage.py makemigrations --dry-run

# Fix diverged migration branches (after a merge conflict)
python manage.py makemigrations --merge
```

> **Best practice:** Always rename migration files to something descriptive immediately after creation, e.g., `0002_add_price_field.py`.

### 6.4 Running Migrations

In [ ]:
# Apply all unapplied migrations
python manage.py migrate

# Apply migrations for one app only
python manage.py migrate my_app

# Migrate to a specific migration (forward or backward)
python manage.py migrate myapp 0005

# Reverse ALL migrations for an app (use with caution!)
python manage.py migrate myapp zero

### 6.5 Squashing Migrations

When you have many migrations in a mature app, you can squash them into a single file:

```bash
# Squash all migrations up to (and including) 0006
python manage.py squashmigrations myapp 0006
```

- Creates a new migration containing all operations
- **Make sure all migrations are applied everywhere before removing old ones**
- The squashed migration replaces the old ones once it has been applied

### 6.6 Custom (Data) Migrations

Sometimes you need to run arbitrary Python code during a migration — e.g., to populate a new column or transform existing data. Use an **empty migration** with `RunPython`:

In [ ]:
# In a migration file (e.g., 0007_populate_slug.py)
from django.db import migrations

def populate_slug(apps, schema_editor):
    # Use apps.get_model() — do NOT import the model class directly
    # (the model might have changed since this migration was written)
    Product = apps.get_model('store', 'Product')
    for product in Product.objects.all():
        product.slug = product.name.lower().replace(' ', '-')
        product.save()

def undo_populate_slug(apps, schema_editor):
    Product = apps.get_model('store', 'Product')
    Product.objects.update(slug='')


class Migration(migrations.Migration):
    dependencies = [('store', '0006_product_slug')]

    operations = [
        migrations.RunPython(
            populate_slug,
            reverse_code=undo_populate_slug  # optional: enables reversal
        )
    ]

---
## 7. Optimizing the ORM

### 7.1 The Golden Rule: Don't Optimize Prematurely

> *Premature optimization is the root of all evil.* — Donald Knuth

Profile first. Fix only measured bottlenecks.

### 7.2 The Most Common Anti-Pattern: Filtering in Python

In [ ]:
# BAD: loads ALL products into Python, then filters in Python
# This is O(n) memory and makes the database do much less work
products_in_stock = []
for p in Product.objects.all():
    if p.stock_count > 0:
        products_in_stock.append(p)

# GOOD: push the filter to the database
products_in_stock = Product.objects.filter(stock_count__gt=0)

**Rule: Always filter, sort, and aggregate in the database, not in Python.**

### 7.3 The N+1 Query Problem and How to Fix It

The N+1 problem: fetching a list of N objects and then accessing a related object on each one, causing N extra queries.

In [ ]:
# BAD — N+1 queries: 1 query for orders + N queries (one per order) for user
orders = Order.objects.all()      # 1 query
for order in orders:
    print(order.user.email)       # 1 extra query per order!


# GOOD — select_related: 1 SQL JOIN, no extra queries
# Use for ForeignKey / OneToOne (single related object)
orders = Order.objects.select_related('user').all()
for order in orders:
    print(order.user.email)       # no extra query, already fetched


# GOOD — prefetch_related: separate query but then Python-side join
# Use for ManyToMany / reverse ForeignKey (multiple related objects)
products = Product.objects.prefetch_related('images').all()
for product in products:
    for image in product.images.all():  # no extra query
        print(image.url)

| Tool | Use case | How it works |
|---|---|---|
| `select_related('field')` | FK / OneToOne (forward) | SQL JOIN — single query |
| `prefetch_related('field')` | ManyToMany / reverse FK | 2 queries + Python join |

### 7.4 QuerySet Caching

In [ ]:
# The evaluated result is cached on the queryset object
products = Product.objects.filter(stock_count__gt=0)

for p in products:   # SQL fires here, results cached
    print(p.name)

for p in products:   # No SQL — uses the cached result
    print(p.price)


# WARNING: Each NEW queryset object has its own cache
# This fires SQL TWICE:
for p in Product.objects.all():   # query 1
    pass
for p in Product.objects.all():   # query 2 (different qs object!)
    pass

### 7.5 Raw SQL Queries

When the ORM isn't enough, Django lets you drop down to raw SQL while still returning model instances:

In [ ]:
from django.db import connection

# Option 1: Manager.raw() — returns model instances
products = Product.objects.raw(
    'SELECT * FROM store_product WHERE stock_count > %s',
    [0]
)

# Option 2: Direct cursor — for non-model queries
with connection.cursor() as cursor:
    cursor.execute('UPDATE store_product SET price = price * 0.9 WHERE on_sale = True')
    rows_updated = cursor.rowcount

### 7.6 Database Transactions with `ATOMIC_REQUESTS`

Wrap each HTTP request in a transaction so partial failures roll back automatically:

In [ ]:
# In settings.py
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        # ...
        'ATOMIC_REQUESTS': True,  # wraps every request in a transaction
    }
}

For fine-grained transaction control, use `atomic()` directly:

In [ ]:
from django.db import transaction

# As a decorator
@transaction.atomic
def place_order(cart):
    order = Order.objects.create(...)
    for item in cart:
        OrderLine.objects.create(order=order, ...)
        item.product.stock_count -= item.quantity
        item.product.save()
    # If any step raises an exception, the whole block is rolled back

# As a context manager
with transaction.atomic():
    # ... all DB operations here are in one transaction
    pass

---
## 8. Putting It All Together — A Complete Example

Here is a realistic `models.py` that applies everything covered in this guide:

In [ ]:
from django.db import models
from django.urls import reverse


class TimestampMixin(models.Model):
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        abstract = True


class Category(TimestampMixin):
    name = models.CharField(max_length=100, unique=True)
    slug = models.SlugField(max_length=100, unique=True)

    class Meta:
        ordering = ['name']
        verbose_name_plural = 'Categories'

    def __str__(self):
        return self.name


class ProductQuerySet(models.QuerySet):
    def in_stock(self):
        return self.filter(stock_count__gt=0)

    def affordable(self, max_price):
        return self.filter(price__lte=max_price)

    def with_images(self):
        return self.prefetch_related('images')


class Product(TimestampMixin):
    name = models.CharField(max_length=200)
    slug = models.SlugField(max_length=200, unique=True)
    sku = models.CharField(max_length=50, unique=True)
    description = models.TextField(blank=True)

    # Always use DecimalField for money!
    price = models.DecimalField(max_digits=10, decimal_places=2)
    discount_percentage = models.IntegerField(default=0)

    stock_count = models.IntegerField(default=0)

    # FK — each product belongs to one category
    category = models.ForeignKey(
        'Category',
        on_delete=models.PROTECT,
        related_name='products',
    )

    objects = ProductQuerySet.as_manager()

    class Meta:
        ordering = ['name']

    def __str__(self):
        return self.name

    def get_absolute_url(self):
        return reverse('product-detail', kwargs={'slug': self.slug})

    @property
    def discounted_price(self):
        if self.discount_percentage:
            return self.price * (1 - self.discount_percentage / 100)
        return self.price

    def is_in_stock(self):
        return self.stock_count > 0


class ProductImage(models.Model):
    product = models.ForeignKey(
        'Product',
        on_delete=models.CASCADE,
        related_name='images',
    )
    image = models.ImageField(upload_to='products/')
    alt_text = models.CharField(max_length=200, blank=True)
    is_primary = models.BooleanField(default=False)

    def __str__(self):
        return f'Image for {self.product.name}'

### Usage examples:

In [ ]:
# Get all in-stock socks under €10, with their images pre-fetched
socks = (
    Product.objects
    .in_stock()
    .affordable(10)
    .filter(category__name='Socks')
    .with_images()
    .select_related('category')
)

# All products in a category, cheapest first
cat = Category.objects.get(slug='socks')
cat.products.all().order_by('price')

# Get a product by slug or return 404
from django.shortcuts import get_object_or_404
product = get_object_or_404(Product, slug='blue-socks')

# Access computed property
print(product.discounted_price)

# All images of a product
for img in product.images.all():
    print(img.image.url)

---
## 9. Quick Reference Cheatsheet

### Field Types

| Field | Use for |
|---|---|
| `CharField(max_length=N)` | Short text (required max_length) |
| `TextField()` | Long text |
| `IntegerField()` | Integers |
| `DecimalField(max_digits, decimal_places)` | Exact decimals / money |
| `FloatField()` | Approximate floats (never for money) |
| `BooleanField()` | True/False |
| `DateField()` | Date only |
| `DateTimeField()` | Date + time |
| `EmailField()` | Email (validated) |
| `URLField()` | URL (validated) |
| `SlugField()` | URL-friendly text |
| `ImageField()` | Image upload |
| `FileField()` | File upload |
| `JSONField()` | JSON data |
| `ForeignKey(Model, on_delete=)` | Many-to-one |
| `OneToOneField(Model, on_delete=)` | One-to-one |
| `ManyToManyField(Model)` | Many-to-many |

### Field Options

| Option | Meaning |
|---|---|
| `null=True` | Allow NULL in DB |
| `blank=True` | Allow empty in forms |
| `default=value` | Default value |
| `unique=True` | DB UNIQUE constraint |
| `db_index=True` | Create DB index |
| `verbose_name='...'` | Human-readable label |
| `help_text='...'` | Form hint text |

### Common QuerySet Methods

| Method | Returns | Notes |
|---|---|---|
| `.all()` | QuerySet | Copy of current qs |
| `.filter(**kwargs)` | QuerySet | Matching rows |
| `.exclude(**kwargs)` | QuerySet | Non-matching rows |
| `.order_by('field')` | QuerySet | Sort ascending; prefix `-` for descending |
| `.distinct()` | QuerySet | Remove duplicates |
| `.values('f1','f2')` | QuerySet of dicts | Dict rows (no model objects) |
| `.values_list('f')` | QuerySet of tuples | Tuple rows |
| `.count()` | int | SQL COUNT |
| `.exists()` | bool | SQL EXISTS |
| `.get(**kwargs)` | Model instance | Exactly one result |
| `.first()` | Model / None | First result or None |
| `.last()` | Model / None | Last result or None |
| `.create(**kwargs)` | Model instance | INSERT + return |
| `.get_or_create(**kwargs)` | (instance, created) | Avoid race conditions |
| `.update(**kwargs)` | int | SQL UPDATE |
| `.delete()` | tuple | SQL DELETE |
| `.select_related('fk')` | QuerySet | SQL JOIN for FK |
| `.prefetch_related('m2m')` | QuerySet | Extra query for M2M |

### Migration Commands

```bash
python manage.py makemigrations        # generate migrations
python manage.py makemigrations --dry-run  # preview without writing
python manage.py showmigrations        # list all with applied status
python manage.py migrate               # apply all
python manage.py migrate myapp 0003    # migrate to specific version
python manage.py migrate myapp zero    # reverse all
python manage.py sqlmigrate myapp 0003 # show SQL for a migration
python manage.py squashmigrations myapp 0010  # squash up to 0010
```